In [ ]:
import sys
sys.path.append('../src')

import copy
import numpy as np
import pandas as pd
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset

from model import SimpleCNN
from partition import dirichlet_partition
from federated import federated_train_with_defense, evaluate
from attack import get_confidence_and_loss, build_attack_dataset, run_mia_attack

# ---------------------------
# FULL RUN SETTINGS
# ---------------------------
transform = transforms.ToTensor()
train_dataset = torchvision.datasets.CIFAR10(root='../data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.CIFAR10(root='../data', train=False, download=True, transform=transform)
train_labels = [label for _, label in train_dataset]

num_clients = 5
num_rounds = 25
alpha_values = [0.1, 0.5, 5.0]
defense_modes = ['none', 'flat', 'adaptive']

test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

results = []

for alpha in alpha_values:
    print(f"\n########## ALPHA = {alpha} ##########")
    client_data = dirichlet_partition(train_labels, num_clients=num_clients, alpha=alpha)

    for mode in defense_modes:
        print(f"\n=== Alpha {alpha} | Defense mode: {mode} ===")

        model = SimpleCNN(num_classes=10)
        model = federated_train_with_defense(
            model, train_dataset, client_data, train_labels,
            num_clients=num_clients, num_rounds=num_rounds, defense_mode=mode
        )

        test_acc = evaluate(model, test_loader)

        member_subset = Subset(train_dataset, client_data[0])
        member_loader = DataLoader(member_subset, batch_size=64, shuffle=False)

        member_conf, member_loss = get_confidence_and_loss(model, member_loader)
        nonmember_conf, nonmember_loss = get_confidence_and_loss(model, test_loader)

        X, y = build_attack_dataset(member_conf, member_loss, nonmember_conf, nonmember_loss)
        attack_metrics = run_mia_attack(X, y)

        print(f"Test accuracy: {test_acc:.4f}")
        print(f"Attack accuracy: {attack_metrics['accuracy']:.4f}")
        print(f"Attack F1: {attack_metrics['f1']:.4f}")

        model_filename = f"../models/model_alpha{alpha}_{mode}.pt"
        torch.save(model.state_dict(), model_filename)

        results.append({
            'alpha': alpha,
            'defense_mode': mode,
            'test_accuracy': test_acc,
            'attack_accuracy': attack_metrics['accuracy'],
            'attack_precision': attack_metrics['precision'],
            'attack_recall': attack_metrics['recall'],
            'attack_f1': attack_metrics['f1']
        })

results_df = pd.DataFrame(results)
results_df.to_csv('../results/experiment_results.csv', index=False)

print("\n\n=== FINAL RESULTS TABLE ===")
print(results_df)

In [3]:
noise_levels = [0.01, 0.05, 0.1, 0.2, 0.3]

for noise in noise_levels:
    model = SimpleCNN(num_classes=10)
    
    # temporarily override the noise value directly for this test
    from defense import apply_dp_flat
    import federated
    
    client_data_test = dirichlet_partition(train_labels, num_clients=5, alpha=0.5)
    
    # quick manual training loop with this noise level
    for round_num in range(10):
        client_state_dicts = []
        for client_id in range(5):
            subset = Subset(train_dataset, client_data_test[client_id])
            dataloader = DataLoader(subset, batch_size=32, shuffle=True)
            
            current_global_state = copy.deepcopy(model.state_dict())
            state_dict = federated.local_train(model, dataloader, epochs=1, lr=0.01)
            state_dict = apply_dp_flat(state_dict, current_global_state, clip_norm=5.0, noise_scale=noise)
            client_state_dicts.append(state_dict)
        
        new_state = federated.federated_average(client_state_dicts)
        model.load_state_dict(new_state)
    
    test_acc = evaluate(model, test_loader)
    
    member_subset = Subset(train_dataset, client_data_test[0])
    member_loader = DataLoader(member_subset, batch_size=64, shuffle=False)
    member_conf, member_loss = get_confidence_and_loss(model, member_loader)
    nonmember_conf, nonmember_loss = get_confidence_and_loss(model, test_loader)
    X, y = build_attack_dataset(member_conf, member_loss, nonmember_conf, nonmember_loss)
    attack_metrics = run_mia_attack(X, y)
    
    print(f"noise={noise} -> test_acc={test_acc:.4f}, attack_acc={attack_metrics['accuracy']:.4f}")

noise=0.01 -> test_acc=0.3710, attack_acc=0.5403


KeyboardInterrupt: 